# `treasury_fiscal(series, start, end=None)` and `treasury_fiscal_series()`

US federal debt, daily, from `indicators.treasury_fiscal`: Treasury's Debt to the Penny, oldest first. The total public debt outstanding, and its two parts, debt held by the public and intragovernmental holdings. `treasury_fiscal_series()` is the map from each series id to its dataset, name, units, frequency and category. Every cell runs against prod, read-only, as `readonly_user`. Change the arguments and re-run any cell.

A series is named by its Treasury field name (`tot_pub_debt_out_amt`) or by its name, in any case and spacing. Days with no value are left out, never filled.

> **About the data shown.** Source: U.S. Department of the Treasury, Fiscal Data (Debt to the Penny), a US Government work in the public domain. One change, by arora_data: the amounts are Treasury's dollar figures divided by 1,000,000, so they are in USD millions.

In [1]:
from datetime import date, timedelta
from decimal import Decimal

import polars as pl
from IPython.display import Markdown

from arora_data.dates import today_et
from arora_data.frames import to_df
from arora_data.sync import treasury_fiscal, treasury_fiscal_series
from arora_data.treasury_fiscal import TreasuryFiscalValue


def show(rows):
    """One result or a list of results as a Markdown table."""
    rows = [r.model_dump() for r in (rows if isinstance(rows, list) else [rows])]
    if not rows:
        return Markdown("0 rows")
    head = list(rows[0])
    lines = ["|".join(head), "|".join("---" for _ in head)]
    lines += ["|".join(str(r[c]).replace("|", "/") for c in head) for r in rows]
    return Markdown(f"{len(rows)} rows\n\n" + "\n".join(lines))

## The series
Every series the table holds, read from it on every call. The labels come from each series' latest row; `first_date` and `latest_date` are its first and latest days with a value.

In [2]:
show(treasury_fiscal_series())

3 rows

series_id|dataset|name|units|frequency|category|first_date|latest_date
---|---|---|---|---|---|---|---
debt_held_public_amt|treasury/debt_to_penny|Debt Held by the Public|USD millions|daily|Fiscal — Debt|1997-09-30|2026-11-19
intragov_hold_amt|treasury/debt_to_penny|Intragovernmental Holdings|USD millions|daily|Fiscal — Debt|1997-09-30|2026-11-19
tot_pub_debt_out_amt|treasury/debt_to_penny|Total Public Debt Outstanding|USD millions|daily|Fiscal — Debt|1993-04-01|2026-11-19

## One series, by id or by name
The total public debt over the last two weeks. The id and the name, in any case and spacing, give the same rows. Each row is a business day: weekends and federal holidays have none.

In [3]:
two_weeks = today_et() - timedelta(days=14)
assert treasury_fiscal("tot_pub_debt_out_amt", two_weeks) == treasury_fiscal("TOT_PUB_DEBT_OUT_AMT", two_weeks) \
    == treasury_fiscal(" total public DEBT outstanding ", two_weeks)
show(treasury_fiscal("tot_pub_debt_out_amt", two_weeks))

8 rows

series_id|name|date|value|units
---|---|---|---|---
tot_pub_debt_out_amt|Total Public Debt Outstanding|2026-11-09|40559796.58924337|USD millions
tot_pub_debt_out_amt|Total Public Debt Outstanding|2026-11-10|40584130.08027289|USD millions
tot_pub_debt_out_amt|Total Public Debt Outstanding|2026-11-12|40608506.85690325|USD millions
tot_pub_debt_out_amt|Total Public Debt Outstanding|2026-11-13|40616678.82709207|USD millions
tot_pub_debt_out_amt|Total Public Debt Outstanding|2026-11-16|40662417.39561743|USD millions
tot_pub_debt_out_amt|Total Public Debt Outstanding|2026-11-17|40714946.31657334|USD millions
tot_pub_debt_out_amt|Total Public Debt Outstanding|2026-11-18|40690769.63884995|USD millions
tot_pub_debt_out_amt|Total Public Debt Outstanding|2026-11-19|40709780.70031702|USD millions

## Units
Amounts are in USD millions: 40709780.7 is about $40.7 trillion. The latest day, in USD trillions:

In [4]:
for s in treasury_fiscal_series():
    v = treasury_fiscal(s.series_id, two_weeks)[-1]
    print(f"{s.name:<32} {v.date}  {(v.value / Decimal(10**6)).quantize(Decimal('0.001')):>8}")

Debt Held by the Public          2026-11-19    32.901


Intragovernmental Holdings       2026-11-19     7.808


Total Public Debt Outstanding    2026-11-19    40.710


## Full history in one call
From 1900 there is no row limit; the total starts on 1993-04-01. The debt at each year's last business day, in USD trillions:

In [5]:
total = to_df(treasury_fiscal("tot_pub_debt_out_amt", date(1900, 1, 1)), TreasuryFiscalValue)
print(f"{total.height:,} days, {total['date'].min()} to {total['date'].max()}")
(total.group_by(pl.col("date").dt.year().alias("year")).agg(pl.col("value").sort_by("date").last())
      .with_columns((pl.col("value") / 1_000_000).round(2).alias("USD trillions")).drop("value").sort("year"))

8,438 days, 1993-04-01 to 2026-11-19


year,USD trillions
i32,"decimal[38,8]"
1993,4.54000000
1994,4.80000000
1995,4.99000000
1996,5.32000000
1997,5.50000000
…,…
2022,31.42000000
2023,34.00000000
2024,36.22000000


## When the parts start
Treasury split the total into its two parts later, and the days before that are NULL and left out. The parts have a value only at fiscal year end from 1997-09-30 to 2000-09-29, at month end from 2001-09-28 to 2005-03-31 (no January 2003), and every day from 2005-04-04. Days with a value, per year:

In [6]:
public = to_df(treasury_fiscal("debt_held_public_amt", date(1900, 1, 1)), TreasuryFiscalValue)
public.filter(pl.col("date") < date(2006, 1, 1)).group_by(pl.col("date").dt.year().alias("year")).len().sort("year")

year,len
i32,u32
1997,1
1998,1
1999,1
2000,1
2001,4
2002,12
2003,11
2004,12
2005,191


## The parts add up to the total, but one day
On every day with both parts, they add up to the total, except 2025-08-04: Treasury's own figures for that day put the parts $10 billion above the total. The values are served as Treasury published them.

In [7]:
day = date(2025, 8, 4)
v = {s.series_id: treasury_fiscal(s.series_id, day, day)[0].value for s in treasury_fiscal_series()}
print(v)
print("parts - total, USD millions:", v["debt_held_public_amt"] + v["intragov_hold_amt"] - v["tot_pub_debt_out_amt"])

{'debt_held_public_amt': Decimal('29523300.14853885'), 'intragov_hold_amt': Decimal('7314864.64937546'), 'tot_pub_debt_out_amt': Decimal('36828164.79791431')}
parts - total, USD millions: 10000.00000000


## Revisions
The main repo re-reads the whole history on every run, so a revised value replaces the earlier one, and earlier values are not stored.

## Not here
Quarterly total public debt is `fred("GFDEBTN")`.

## Bad input
An unknown id or name, or a start after the end, raises a clear error before any values are read.

In [8]:
for args in [("GFDEBTN", date(2026, 1, 1)), ("total public debt", date(2026, 1, 1)), (None, date(2026, 1, 1)),
             ("tot_pub_debt_out_amt", date(2026, 1, 2), date(2026, 1, 1))]:
    try:
        treasury_fiscal(*args)
    except ValueError as e:
        print(e)

'GFDEBTN' is not a stored US Treasury fiscal series: use an id or name from treasury_fiscal_series(), such as tot_pub_debt_out_amt or 'Debt Held by the Public'


'total public debt' is not a stored US Treasury fiscal series: use an id or name from treasury_fiscal_series(), such as tot_pub_debt_out_amt or 'Debt Held by the Public'


None is not a stored US Treasury fiscal series: use an id or name from treasury_fiscal_series(), such as tot_pub_debt_out_amt or 'Debt Held by the Public'
start must be on or before end
